Дорогой студент!

В домашнем задании Ultra Pro занятия по обработке тектсов с помощью НС мы ставим задачу распознать уже не 6, как ранее, а целых 20 русских писателей! Это подразумевает и больший размер базы для обучения соответственно. Ячейка для скачивания базы уже включена в ноутбук задания.


 В задании необходимо выполнить следующие пункты:

  1. Загрузить саму базу по ссылке и подговить файлы базы для обработки.
  2. Создать обучающую и проверочную выборки, обратив особое внимание на балансировку базы: количество примеров каждого класса должно быть примерно одного порядка. При этом для разбивки необходимо применить цикл. Проверочная выборка должна быть 20% от общей выборки.
  3. Подготовьте выборки для обучения и обучите сеть. Добейтесь результата точности сети не менее 95% на проверочной выборке модели Bag of Words и 75-80% - для модели Embedding.
   


## 1. Скачивание и распаковка архива с текстами 20 писателей

In [5]:
import gdown
from zipfile import ZipFile
import os

# Скачивание архива
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l7/20writers.zip', None, quiet=True)

# Распаковка
with ZipFile("20writers.zip", 'r') as zip_ref:
    zip_ref.extractall("20writers")

## 2. Импорт библиотек

In [6]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Embedding, LSTM, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

%matplotlib inline

## 3. Загрузка текстов 20 писателей и формирование списка классов

In [7]:
def load_writers_data(data_dir):
    texts = []
    class_names = []
    files = os.listdir(data_dir)
    files.sort()
    for fname in files:
        if fname.endswith(".txt"):
            author = fname.replace(".txt", "")
            class_names.append(author)
            with open(os.path.join(data_dir, fname), 'r', encoding='utf-8') as f:
                text = f.read().replace('\n', ' ')
            texts.append(text)
    return texts, class_names

data_dir = "20writers"
texts, class_names = load_writers_data(data_dir)

print(f"Загружено файлов: {len(texts)}")
print(f"Классы: {class_names[:5]}...")
print(f"Всего классов: {len(class_names)}")

Загружено файлов: 20
Классы: ['Беляев', 'Булгаков', 'Васильев', 'Гоголь', 'Гончаров']...
Всего классов: 20


## 4. Балансировка выборки: разбиение каждого текста на фрагменты (цикл по классам)

In [8]:
CHUNK_SIZE = 1000
STEP = 500

def split_text_into_chunks(text, chunk_size, step):
    chunks = []
    for start in range(0, len(text) - chunk_size + 1, step):
        chunks.append(text[start:start + chunk_size])
    return chunks

X_chunks = []
y_chunks = []

for class_id, full_text in enumerate(texts):
    chunks = split_text_into_chunks(full_text, CHUNK_SIZE, STEP)
    X_chunks.extend(chunks)
    y_chunks.extend([class_id] * len(chunks))
    print(f"{class_names[class_id]:20s} → {len(chunks)} фрагментов")

print(f"\nВсего фрагментов: {len(X_chunks)}")
print(f"Классов: {len(set(y_chunks))}")

Беляев               → 4509 фрагментов
Булгаков             → 4001 фрагментов
Васильев             → 5904 фрагментов
Гоголь               → 3929 фрагментов
Гончаров             → 6208 фрагментов
Горький              → 5045 фрагментов
Грибоедов            → 1937 фрагментов
Достоевский          → 10356 фрагментов
Каверин              → 3984 фрагментов
Катаев               → 5157 фрагментов
Куприн               → 4656 фрагментов
Лермонтов            → 3944 фрагментов
Лесков               → 4303 фрагментов
Носов                → 4797 фрагментов
Пастернак            → 6177 фрагментов
Пушкин               → 6815 фрагментов
Толстой              → 6713 фрагментов
Тургенев             → 3959 фрагментов
Чехов                → 13222 фрагментов
Шолохов              → 6771 фрагментов

Всего фрагментов: 112387
Классов: 20


## 5. Разделение на обучающую (80%) и тестовую (20%) выборки

In [9]:
# Разделение
X_train_texts, X_test_texts, y_train, y_test = train_test_split(
    X_chunks, y_chunks,
    test_size=0.2,
    random_state=42,
    stratify=y_chunks
)

# Преобразуем в numpy
y_train = np.array(y_train)
y_test = np.array(y_test)

print(f"Обучающая выборка: {len(X_train_texts)} фрагментов")
print(f"Тестовая выборка:  {len(X_test_texts)} фрагментов")
print(f"Классов в train: {len(np.unique(y_train))}")
print(f"Классов в test:  {len(np.unique(y_test))}")

Обучающая выборка: 89909 фрагментов
Тестовая выборка:  22478 фрагментов
Классов в train: 20
Классов в test:  20


## 6. Bag of Words векторизация и обучение нейросети

In [10]:
from sklearn.feature_extraction.text import CountVectorizer
import gc
import numpy as np

MAX_BOW_FEATURES = 4000

vectorizer = CountVectorizer(
    max_features=MAX_BOW_FEATURES,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.8,
    stop_words=None
)
X_train_bow = vectorizer.fit_transform(X_train_texts)
X_test_bow = vectorizer.transform(X_test_texts)

X_train_bow = X_train_bow.toarray()
X_test_bow = X_test_bow.toarray()

gc.collect()

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

model_bow = Sequential()
model_bow.add(Dense(256, activation='relu', input_shape=(X_train_bow.shape[1],)))
model_bow.add(Dropout(0.4))
model_bow.add(BatchNormalization())
model_bow.add(Dense(128, activation='relu'))
model_bow.add(Dropout(0.4))
model_bow.add(BatchNormalization())
model_bow.add(Dense(64, activation='relu'))
model_bow.add(Dropout(0.3))
model_bow.add(Dense(len(class_names), activation='softmax'))

model_bow.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, mode='max')

history_bow = model_bow.fit(
    X_train_bow, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_test_bow, y_test),
    callbacks=[early_stop],
    verbose=1
)

test_loss_bow, test_acc_bow = model_bow.evaluate(X_test_bow, y_test, verbose=0)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/30
1405/1405 ━━━━━━━━━━━━━━━━━━━━ 18s 9ms/step - accuracy: 0.5131 - loss: 1.6585 - val_accuracy: 0.8039 - val_loss: 0.6740
Epoch 2/30
1405/1405 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.7509 - loss: 0.8448 - val_accuracy: 0.8525 - val_loss: 0.5019
Epoch 3/30
1405/1405 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.8122 - loss: 0.6402 - val_accuracy: 0.8690 - val_loss: 0.4316
Epoch 4/30
1405/1405 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8477 - loss: 0.5179 - val_accuracy: 0.8834 - val_loss: 0.3844
Epoch 5/30
1405/1405 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.8688 - loss: 0.4464 - val_accuracy: 0.8872 - val_loss: 0.3679
Epoch 6/30
1405/1405 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.8827 - loss: 0.3959 - val_accuracy: 0.8913 - val_loss: 0.3562
Epoch 7/30
1405/1405 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8948 - loss: 0.3561 - val_accuracy: 0.8959 - val_loss: 0.3381
Epoch 8/30
1405/1405 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9044 - loss: 0.3214 

## 7. Embedding + Flatten модель

In [12]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

MAX_WORDS = 15000
MAX_LEN = 200

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_texts)

X_train_seq = tokenizer.texts_to_sequences(X_train_texts)
X_test_seq = tokenizer.texts_to_sequences(X_test_texts)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

model_emb = Sequential()
model_emb.add(Embedding(MAX_WORDS, 64, input_length=MAX_LEN))
model_emb.add(Flatten())
model_emb.add(Dense(128, activation='relu'))
model_emb.add(Dropout(0.4))
model_emb.add(BatchNormalization())
model_emb.add(Dense(64, activation='relu'))
model_emb.add(Dropout(0.3))
model_emb.add(Dense(len(class_names), activation='softmax'))

model_emb.build(input_shape=(None, MAX_LEN))

model_emb.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model_emb.summary()

early_stop = EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True, mode='max')

history_emb = model_emb.fit(
    X_train_pad, y_train,
    epochs=20,
    batch_size=128,
    validation_data=(X_test_pad, y_test),
    callbacks=[early_stop],
    verbose=1
)

test_loss_emb, test_acc_emb = model_emb.evaluate(X_test_pad, y_test, verbose=0)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 64)        │       960,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12800)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │     1,638,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 20)             │         1,300 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,608,596 (9.95 MB)

 Trainable params: 2,608,340 (9.95 MB)

 Non-trainable params: 256 (1.00 KB)

Epoch 1/20
703/703 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.3579 - loss: 2.0713 - val_accuracy: 0.6719 - val_loss: 0.9971
Epoch 2/20
703/703 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7303 - loss: 0.8361 - val_accuracy: 0.8030 - val_loss: 0.6237
Epoch 3/20
703/703 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8638 - loss: 0.4263 - val_accuracy: 0.8017 - val_loss: 0.7044
Epoch 4/20
703/703 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9201 - loss: 0.2506 - val_accuracy: 0.8559 - val_loss: 0.5204
Epoch 5/20
703/703 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9460 - loss: 0.1703 - val_accuracy: 0.8572 - val_loss: 0.5100
Epoch 6/20
703/703 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9580 - loss: 0.1336 - val_accuracy: 0.8547 - val_loss: 0.5340
Epoch 7/20
703/703 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9652 - loss: 0.1107 - val_accuracy: 0.8672 - val_loss: 0.5158
Epoch 8/20
703/703 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9714 - loss: 0.0927 - val_accuracy: 0

## 8. Итоговая точность обеих моделей

In [13]:
print(f"Bag of Words точность:     {test_acc_bow:.2%}")
print(f"Embedding точность:        {test_acc_emb:.2%}")

Bag of Words точность:     91.64%
Embedding точность:        87.67%
